<a href="https://colab.research.google.com/github/vivianlinnn/DS41_IDXExchange/blob/main/src/07_LightGBM_Team.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# START : Anjali Manju Gowda

# Install LightGBM
This cell installs the LightGBM library, which is a gradient boosting framework optimized for speed and performance.

In [ ]:
!pip install lightgbm

# Import Required Libraries
We import core Python libraries for data handling, numerical computation, and model evaluation:
- `pandas` & `numpy` → data manipulation and numerical operations
- `LGBMRegressor` → LightGBM gradient boosting regression model
- `r2_score` & `mean_absolute_percentage_error` → metrics to evaluate model performance

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# Load Training and Test Data
We load the cleaned feature-engineered CSV datasets:
- `train_cleaned_fe.csv` → training set
- `test_cleaned_fe.csv` → testing set
The `on_bad_lines='skip'` ensures malformed rows are ignored.

In [ ]:
# Load CSV files
train = pd.read_csv('/content/train_cleaned_fe.csv', engine='python', on_bad_lines='skip')
test  = pd.read_csv('/content/test_cleaned_fe.csv', engine='python', on_bad_lines='skip')

# Convert Columns to Numeric
Ensures that numeric columns are properly typed to prevent errors during modeling. Any non-numeric or invalid entries are converted to NaN.

In [ ]:
# Convert numeric columns safely
numeric_cols = [
    'LivingArea','BedroomsTotal','LotSizeSquareFeet',
    'BathroomsTotalInteger','Bed_Bath_Ratio',
    'Property_Age','Months_From_Dec_2025'
]
for col in numeric_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col]  = pd.to_numeric(test[col], errors='coerce')

train['ClosePrice'] = pd.to_numeric(train['ClosePrice'], errors='coerce')
test['ClosePrice']  = pd.to_numeric(test['ClosePrice'], errors='coerce')

# Remove Rows with Missing Target
Rows where the target variable `ClosePrice` is missing are dropped to ensure the model only trains on valid data.

In [ ]:
# Drop rows with missing target
train = train.dropna(subset=['ClosePrice'])
test  = test.dropna(subset=['ClosePrice'])

# Log-transform the Target Variable
The `ClosePrice` is right-skewed. Taking `log(ClosePrice)`:
- Stabilizes variance
- Reduces influence of extremely high-priced properties
- Improves model performance

In [ ]:
# Log-transform the target
train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice']  = np.log(test['ClosePrice'])

# Create Additional Features
We generate new features to improve predictive power:
- Replace zeros with NaN for Bedrooms and LotSize to prevent division errors
- `Sqft_Per_Bedroom` → living area per bedroom
- `Lot_Utilization` → efficiency of land usage (living area / lot size)

In [ ]:
# Feature engineering (example)
for df in [train, test]:
    df['BedroomsTotal'] = df['BedroomsTotal'].replace(0, np.nan)
    df['LotSizeSquareFeet'] = df['LotSizeSquareFeet'].replace(0, np.nan)
    df['Sqft_Per_Bedroom'] = df['LivingArea'] / df['BedroomsTotal']
    df['Lot_Utilization']  = df['LivingArea'] / df['LotSizeSquareFeet']

# Encode Location Using Median Price per ZIP
Instead of one-hot encoding ZIP codes, we use the median log price for each ZIP:
- Captures neighborhood-level pricing effects
- Reduces high dimensionality
- Unseen ZIPs in test set are assigned the global median

In [ ]:
# ZIP-level median price encoding
zip_median = train.groupby('PostalCode')['LogClosePrice'].median()
train['ZipMedianPrice'] = train['PostalCode'].map(zip_median)
test['ZipMedianPrice']  = test['PostalCode'].map(zip_median)
global_median = train['LogClosePrice'].median()
test['ZipMedianPrice'] = test['ZipMedianPrice'].fillna(global_median)

# Define Model Features
The `tree_features` list includes:
- Location (`ZipMedianPrice`)
- Property attributes (`LivingArea`, `BathroomsTotalInteger`, `BedroomsTotal`)
- Engineered features (`Bed_Bath_Ratio`, `Sqft_Per_Bedroom`, `Lot_Utilization`)
- Time/age features (`Property_Age`, `Months_From_Dec_2025`)

In [ ]:
# Select features
tree_features = [
    'ZipMedianPrice','LivingArea','BathroomsTotalInteger',
    'BedroomsTotal','Bed_Bath_Ratio','Property_Age',
    'Months_From_Dec_2025','Sqft_Per_Bedroom','Lot_Utilization'
]

# Handle Missing and Infinite Values
Ensures all feature columns contain valid numerical values:
- Replace `inf` and `-inf` with NaN
- Drop rows with missing values in features or target

In [ ]:
# Drop any remaining missing values
for df in [train, test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(subset=tree_features + ['LogClosePrice'], inplace=True)


# Prepare Feature Matrices and Target Vectors
Split data into:
- `X_train` / `X_test` → feature matrices
- `y_train` / `y_test` → target variable (`LogClosePrice`)

In [ ]:
# Prepare train/test matrices
X_train = train[tree_features]
y_train = train['LogClosePrice']
X_test  = test[tree_features]
y_test  = test['LogClosePrice']

# Train LightGBM Regressor
- Gradient boosting model (`LGBMRegressor`)
- Parameters:
  - `n_estimators=200` → number of trees
  - `max_depth=6` → max depth per tree to avoid overfitting
  - `learning_rate=0.1` → step size for each boosting iteration
- Trains a sequence of decision trees to minimize prediction error

In [ ]:
# Train LightGBM
lgbm_model = LGBMRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)
lgbm_model.fit(X_train, y_train)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001921 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1236
[LightGBM] [Info] Number of data points in the train set: 67669, number of used features: 9
[LightGBM] [Info] Start training from score 13.773877
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

LGBMRegressor(max_depth=6, n_estimators=200, random_state=42)

# Generate Predictions
Predict log-transformed prices for:
- Training set (`train_pred`)
- Testing set (`test_pred`)

In [ ]:
# Predictions
train_pred = lgbm_model.predict(X_train)
test_pred  = lgbm_model.predict(X_test)


# Convert Predictions Back to Dollar Scale
Exponentiate log predictions to return to actual price scale for evaluation:
- `y_train_d`, `y_test_d` → original target
- `train_pred_d`, `test_pred_d` → predicted prices

In [ ]:
# Convert back to original scale
y_train_d = np.exp(y_train)
y_test_d  = np.exp(y_test)
train_pred_d = np.exp(train_pred)
test_pred_d  = np.exp(test_pred)

# Evaluate Model Performance
Metrics used:
- `R²` → proportion of variance explained
- `MAPE` → mean absolute percentage error
- `MdAPE` → median absolute percentage error (robust to outliers)
This gives an understanding of both accuracy and reliability of predictions.

In [ ]:
# Metrics
train_r2 = r2_score(y_train, train_pred)
test_r2  = r2_score(y_test, test_pred)
train_mape = mean_absolute_percentage_error(y_train_d, train_pred_d) * 100
test_mape  = mean_absolute_percentage_error(y_test_d, test_pred_d) * 100
train_mdape = np.median(np.abs((y_train_d - train_pred_d) / y_train_d)) * 100
test_mdape  = np.median(np.abs((y_test_d - test_pred_d) / y_test_d)) * 100

print("LightGBM Results")
print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
print(f"Train MAPE: {train_mape:.2f}%, Test MAPE: {test_mape:.2f}%")
print(f"Train MdAPE: {train_mdape:.2f}%, Test MdAPE: {test_mdape:.2f}%")

LightGBM Results
Train R²: 0.9159, Test R²: 0.8903
Train MAPE: 13.24%, Test MAPE: 14.84%
Train MdAPE: 9.69%, Test MdAPE: 10.50%


# LightGBM Model Performance Summary

The LightGBM regression model was trained on log-transformed property prices using feature-engineered variables, including ZIP-level median prices, property attributes, and derived metrics such as `Sqft_Per_Bedroom` and `Lot_Utilization`.

## Key Evaluation Metrics

| Metric       | Training Set | Testing Set |
|-------------|-------------|------------|
| **R²**      | 0.916       | 0.890      |
| **MAPE (%)**| 13.24       | 14.84      |
| **MdAPE (%)**| 9.69       | 10.50      |

### Interpretation

1. **High R²**: The model explains ~92% of variance on training data and ~89% on unseen test data, indicating strong predictive power and reasonable generalization.  
2. **Low MAPE and MdAPE**: The mean and median absolute percentage errors are low, meaning predictions are, on average, within ~15% of the true property prices, with typical errors closer to 10%.  
3. **Robustness**: The small difference between training and testing metrics suggests the model is not overfitting and handles unseen data well.  
4. **Practical Use**: This LightGBM model effectively captures non-linear relationships between property features and prices, making it suitable for real estate price estimation and decision-making.

> Overall, the LightGBM model provides an accurate, interpretable, and robust approach for predicting residential property closing prices using feature-engineered data.

# END : Anjali Manju Gowda